In [8]:
import pandas as pd


In [9]:
# Read in RASMI material intensity data and MI from old IMAGE-Materials CP
mi_rasmi = pd.read_excel("MI_ranges_20230905.xlsx", index_col=[0, 1, 2, 3, 4], 
                         sheet_name=["concrete", "brick", "wood", "steel", "glass", "plastics", "aluminum", "copper"])

mi_image_mat = pd.read_csv("Building_materials_old.csv", index_col=[0, 1, 2])

mi_image_mat_commercial = pd.read_csv("materials_commercial_old.csv", index_col=[0, 1, 2])


In [ ]:
# dictionary that maps RASMI R32 Regions to IMAGE R26 Regions

image_to_rasmi = {
    1: 'OECD_CAN',
    2: 'OECD_USA',
    3: 'LAM_MEX',
    4: 'LAM_LAM-L',
    5: 'LAM_BRA',
    6: 'LAM_LAM-L',
    7: ['MAF_MEA-H', 'MAF_MEA-M', 'MAF_NAF'],
    8: ['MAF_SSA-L', 'MAF_SSA-M'],
    9: ['MAF_SSA-L', 'MAF_SSA-M'],
    10: 'MAF_SAF',
    11: ['OECD_EFTA', 'OECD_EU12-H', 'OECD_EU12-M', 'OECD_EU15'],
    12: 'REF_EEU-FSU',
    13: 'OECD_TUR',
    14: ['REF_EEU-FSU', 'ASIA_TWN'],
    15: ['OECD_EEU', 'REF_CAS'],
    16: 'REF_RUS',
    17: ['MAF_MEA-H', 'MAF_MEA-M'],
    18: ['ASIA_IND', 'ASIA_OAS-CPA', 'ASIA_PAK'],
    19: 'OECD_KOR',
    20: 'ASIA_CHN',
    21: ['ASIA_OAS-L', 'ASIA_OAS-M'],
    22: 'ASIA_IDN',
    23: 'OECD_JPN',
    24: 'OECD_AUNZ',
    25: ['ASIA_OAS-L', 'ASIA_OAS-M'],
    26: ['MAF_SSA-L', 'MAF_SSA-M']
}

housing_type_image_to_rasmi = {
    1 : 'RS', #detached house to residential single family
    2 : 'RS', # semi detached house to residential single family
    3 : 'RM', # appartement to residential multi-family
    4 : 'RM', # high-rise to residential multi-family
}

housing_type_rasmi_to_image = {
    'RS' : [1, 2], # detached house and semi detached house to
    'RM' : [3, 4]  # appartement and high-rise to
}

housing_type_to_rasmi_building_structure = {
    1 : ['C', 'M', 'S', 'T'], # assumption that detached housing are average all structures
    2 : ['C', 'M', 'S', 'T'], # assumption that semi detached housing are average all structures
    3 : ['C', 'S'], # assumption that appartement are only made out of cement and steel structures
    4 : ['S'] # assumption that high-rise only steel structures
}

material_list_rasmi = ["steel", "concrete", "wood", "copper", "aluminum", "glass", "brick"]
mis_list_target = ["Steel", "Concrete", "Wood", "Copper", "Aluminium", "Glass", "Brick"]

# regions where total steel consumption is higher than what is estimated by image-materials sectors buildings & vehicles
steel_regions_adapt = [4, 8, 9, 22, 25, 26]

In [23]:


def replace_old_mis_with_rasmi(mi_rasmi: pd.DataFrame, material_list_rasmi: list, start_year: int = 2020, 
                               target_year: int = 2050, data_value = "p_50"):
    """    
    Replace old material intensities in IMAGE-Materials with RASMI values for concrete.
    
    material_name: str, non capitalised from rasmi
    mis_list_target_name: str, capitalised name of the material intensity in IMAGE-Materials
    data_value: str, can also be 0, 25, 75 or 100 percentile
    """
    for material_name in material_list_rasmi:
        # Capitalize material name
        material_name_capitalized = material_name.capitalize()
        # Replace if material_name is aluminum to Aluminium because of different spelling
        if material_name == "aluminum":
            material_name_capitalized = "Aluminium"

        print(material_name_capitalized, material_name)
        material_intensities = mi_rasmi.get(material_name)

        # loop through all IMAGE classes and the according rasmi regions
        for image_region, rasmi_region in image_to_rasmi.items():
            # convert str to list is necessary:
            if isinstance(rasmi_region, str):
                rasmi_region = [rasmi_region]
            
            # loop through IMAGE regions and get the mean concrete mi value for each region for RS and RM (housing types)
            for housingtype_image, housingtype_rasmi in housing_type_image_to_rasmi.items():

                if image_region in steel_regions_adapt and material_name == "steel":
                    data_value = "p_25"
                else:
                    pass

                filtered_mis = material_intensities[material_intensities.index.get_level_values('R5_32').isin(rasmi_region) #filter for the right region
                                        & material_intensities.index.get_level_values('function').isin([housingtype_rasmi]) # filter for the right housing type of rasmi
                                        & material_intensities.index.get_level_values('structure').isin(housing_type_to_rasmi_building_structure[housingtype_image])].loc[:, data_value] # filter for the right building structure

                mean_mi_value = filtered_mis.mean()
                # replace old values with new values
                mi_image_mat.loc[([start_year, target_year], image_region, housingtype_image), material_name_capitalized] = mean_mi_value
                # save as csv
    mi_image_mat.to_csv("Building_materials_rasmi.csv")
    print("done")

replace_old_mis_with_rasmi(mi_rasmi, material_list_rasmi)


Steel steel
Concrete concrete
Wood wood
Copper copper
Aluminium aluminum
Glass glass
Brick brick
done


In [26]:
def replace_old_mis_with_rasmi_commercial(mi_rasmi: pd.DataFrame, material_list_rasmi: list, start_year: int = 2020, 
                               target_year: int = 2050, data_value = "p_50"):
    """    
    Replace old material intensities in IMAGE-Materials with RASMI values for concrete.
    
    material_name: str, non capitalised from rasmi
    mis_list_target_name: str, capitalised name of the material intensity in IMAGE-Materials
    data_value: str, can also be 0, 25, 75 or 100 percentile
    """
    for material_name in material_list_rasmi:
        # Capitalize material name
        material_name_capitalized = material_name.capitalize()
        # Replace if material_name is aluminum to Aluminium because of different spelling
        if material_name == "aluminum":
            material_name_capitalized = "Aluminium"

        print(material_name_capitalized, material_name)
        material_intensities = mi_rasmi.get(material_name)

        # loop through all IMAGE classes and the according rasmi regions
        for image_region, rasmi_region in image_to_rasmi.items():
            # convert str to list is necessary:
            if isinstance(rasmi_region, str):
                rasmi_region = [rasmi_region]

            filtered_mis = material_intensities[material_intensities.index.get_level_values('R5_32').isin(rasmi_region) #filter for the right region
                                        & material_intensities.index.get_level_values('function').isin(['NR']) # filter for the right housing type of rasmi
                                        & material_intensities.index.get_level_values('structure').isin(['C', 'M', 'S', 'T'])].loc[:, data_value]
            print(filtered_mis)

    mi_image_mat_commercial.to_csv("Building_materials_rasmi.csv")
    print("done")
    

In [27]:

replace_old_mis_with_rasmi_commercial(mi_rasmi, material_list_rasmi)

Steel steel
     function  structure  R5_32     R5  
19   NR        C          OECD_CAN  OECD     46.000000
51   NR        M          OECD_CAN  OECD     23.597015
83   NR        S          OECD_CAN  OECD    142.000000
115  NR        T          OECD_CAN  OECD     29.023165
Name: p_50, dtype: float64
     function  structure  R5_32     R5  
28   NR        C          OECD_USA  OECD     53.900000
60   NR        M          OECD_USA  OECD     23.597015
92   NR        S          OECD_USA  OECD    142.000000
124  NR        T          OECD_USA  OECD     29.023165
Name: p_50, dtype: float64
     function  structure  R5_32    R5 
11   NR        C          LAM_MEX  LAM     54.681259
43   NR        M          LAM_MEX  LAM     26.000000
75   NR        S          LAM_MEX  LAM    142.000000
107  NR        T          LAM_MEX  LAM      1.500000
Name: p_50, dtype: float64
     function  structure  R5_32      R5 
9    NR        C          LAM_LAM-L  LAM     54.681259
41   NR        M          LAM_LAM-L  L

In [ ]:
mi_image_mat_commercial.loc[([start_year, target_year], image_region, housingtype_image), material_name_capitalized] = mean_mi_value

,,Retail+,Hotels+,Govt+
Material,Offices,,,
Steel,104.7,71.2,80.0,86.9
Concrete,851.8,771.4,723.2,934.6
Aluminium,4.8,2.4,4.4,5.8
Copper,3.9,2.3,3.5,3.4
Wood,7.7,18.3,25.7,17.2
Glass,5.3,4.3,3.6,9.6
Cement,204.7,269.9,786.8,409.3
Brick,379.5,465.8,357.9,387.8


In [15]:
# replace material intensities for commercial buildings
